# Ollama Quran Chunk Benchmark

Objective: compare available Ollama text models and target chunk sizes for Quran enrichment before SurrealDB insertion.

Success criteria:
- chunk has surrounding context but model returns metadata for target chunk only;
- output is valid JSON for category, entities, relations, and summary;
- latency supports a practical whole-Quran ingestion estimate;
- result suggests model + chunk size for first SurrealDB pipeline.

## Method

Each test builds overlapping chunks from Quran sample passages. Prompt contains previous context, target chunk, and next context. Persisted record should keep target text plus context refs/metadata; relations and categories come from target only.

Benchmark uses small sample for speed. For production, rerun with more passages and set `BENCHMARK_MODELS` / `CHUNK_SIZES` env vars.

In [1]:
from __future__ import annotations

import json
import math
import os
import re
import statistics
import time
from dataclasses import dataclass
from pathlib import Path
from typing import Any

import requests

NOTEBOOK_ROOT = Path.cwd()
OUTPUT_DIR = (NOTEBOOK_ROOT / "output") if NOTEBOOK_ROOT.name == "experiments" else (NOTEBOOK_ROOT / "experiments" / "output")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

OLLAMA_URL = (os.environ.get("OLLAMA_URL") or os.environ.get("OLLAMA_HOST") or "").rstrip("/")
QURAN_WORD_ESTIMATE = int(os.environ.get("QURAN_WORD_ESTIMATE", "77430"))
DEFAULT_MODELS = ["llama3.2:3b", "qwen2.5:7b", "gemma2:9b", "aya:latest"]
DEFAULT_CHUNK_SIZES = [80, 160, 260]
MODEL_TIMEOUT_SECONDS = int(os.environ.get("MODEL_TIMEOUT_SECONDS", "240"))

configured_models = os.environ.get("BENCHMARK_MODELS", "")
BENCHMARK_MODELS = [m.strip() for m in configured_models.split(",") if m.strip()] or DEFAULT_MODELS

configured_sizes = os.environ.get("CHUNK_SIZES", "")
CHUNK_SIZES = [int(s.strip()) for s in configured_sizes.split(",") if s.strip()] or DEFAULT_CHUNK_SIZES

if not OLLAMA_URL:
    raise RuntimeError("Set OLLAMA_URL or OLLAMA_HOST in Docker Compose env.")

print("Notebook root:", NOTEBOOK_ROOT)
print("Ollama URL set:", bool(OLLAMA_URL))
print("Candidate models:", BENCHMARK_MODELS)
print("Chunk sizes:", CHUNK_SIZES)

Notebook root: /app/notebooks/experiments
Ollama URL set: True
Candidate models: ['llama3.2:3b', 'qwen2.5:7b']
Chunk sizes: [80, 160]


In [2]:
def ollama_post(path: str, payload: dict[str, Any], timeout: int = MODEL_TIMEOUT_SECONDS) -> dict[str, Any]:
    response = requests.post(f"{OLLAMA_URL}{path}", json=payload, timeout=timeout)
    response.raise_for_status()
    return response.json()


def list_ollama_models() -> list[str]:
    response = requests.get(f"{OLLAMA_URL}/api/tags", timeout=20)
    response.raise_for_status()
    return sorted(model["name"] for model in response.json().get("models", []))


available_models = list_ollama_models()
models = [model for model in BENCHMARK_MODELS if model in available_models]
missing_models = [model for model in BENCHMARK_MODELS if model not in available_models]

print(f"Available model count: {len(available_models)}")
print("Selected models:", models)
print("Missing configured models:", missing_models)

if not models:
    raise RuntimeError("No configured benchmark models found in Ollama /api/tags.")

Available model count: 25
Selected models: ['llama3.2:3b', 'qwen2.5:7b']
Missing configured models: []


## Quran Sample Corpus

Sample mixes short surahs, Ayat al-Kursi, end of Al-Baqarah, and Yasin opening. It is enough to test JSON behavior, category stability, and relation extraction, but not enough for final model selection without a broader eval set.

In [3]:
SAMPLE_AYAHS = [
    ("1:1", "بسم الله الرحمن الرحيم"),
    ("1:2", "الحمد لله رب العالمين"),
    ("1:3", "الرحمن الرحيم"),
    ("1:4", "مالك يوم الدين"),
    ("1:5", "إياك نعبد وإياك نستعين"),
    ("1:6", "اهدنا الصراط المستقيم"),
    ("1:7", "صراط الذين أنعمت عليهم غير المغضوب عليهم ولا الضالين"),
    ("2:255", "الله لا إله إلا هو الحي القيوم لا تأخذه سنة ولا نوم له ما في السماوات وما في الأرض من ذا الذي يشفع عنده إلا بإذنه يعلم ما بين أيديهم وما خلفهم ولا يحيطون بشيء من علمه إلا بما شاء وسع كرسيه السماوات والأرض ولا يؤوده حفظهما وهو العلي العظيم"),
    ("2:285", "آمن الرسول بما أنزل إليه من ربه والمؤمنون كل آمن بالله وملائكته وكتبه ورسله لا نفرق بين أحد من رسله وقالوا سمعنا وأطعنا غفرانك ربنا وإليك المصير"),
    ("2:286", "لا يكلف الله نفسا إلا وسعها لها ما كسبت وعليها ما اكتسبت ربنا لا تؤاخذنا إن نسينا أو أخطأنا ربنا ولا تحمل علينا إصرا كما حملته على الذين من قبلنا ربنا ولا تحملنا ما لا طاقة لنا به واعف عنا واغفر لنا وارحمنا أنت مولانا فانصرنا على القوم الكافرين"),
    ("36:1", "يس"),
    ("36:2", "والقرآن الحكيم"),
    ("36:3", "إنك لمن المرسلين"),
    ("36:4", "على صراط مستقيم"),
    ("36:5", "تنزيل العزيز الرحيم"),
    ("36:6", "لتنذر قوما ما أنذر آباؤهم فهم غافلون"),
    ("36:7", "لقد حق القول على أكثرهم فهم لا يؤمنون"),
    ("36:8", "إنا جعلنا في أعناقهم أغلالا فهي إلى الأذقان فهم مقمحون"),
    ("36:9", "وجعلنا من بين أيديهم سدا ومن خلفهم سدا فأغشيناهم فهم لا يبصرون"),
    ("36:10", "وسواء عليهم أأنذرتهم أم لم تنذرهم لا يؤمنون"),
    ("36:11", "إنما تنذر من اتبع الذكر وخشي الرحمن بالغيب فبشره بمغفرة وأجر كريم"),
    ("36:12", "إنا نحن نحيي الموتى ونكتب ما قدموا وآثارهم وكل شيء أحصيناه في إمام مبين"),
    ("112:1", "قل هو الله أحد"),
    ("112:2", "الله الصمد"),
    ("112:3", "لم يلد ولم يولد"),
    ("112:4", "ولم يكن له كفوا أحد"),
    ("113:1", "قل أعوذ برب الفلق"),
    ("113:2", "من شر ما خلق"),
    ("113:3", "ومن شر غاسق إذا وقب"),
    ("113:4", "ومن شر النفاثات في العقد"),
    ("113:5", "ومن شر حاسد إذا حسد"),
    ("114:1", "قل أعوذ برب الناس"),
    ("114:2", "ملك الناس"),
    ("114:3", "إله الناس"),
    ("114:4", "من شر الوسواس الخناس"),
    ("114:5", "الذي يوسوس في صدور الناس"),
    ("114:6", "من الجنة والناس"),
]


def tokenize_arabic(text: str) -> list[str]:
    return re.findall(r"[\u0600-\u06FF]+", text)


flat_words: list[dict[str, str]] = []
for ref, text in SAMPLE_AYAHS:
    for word in tokenize_arabic(text):
        flat_words.append({"ref": ref, "word": word})

print("Sample ayahs:", len(SAMPLE_AYAHS))
print("Sample words:", len(flat_words))
print("Whole Quran word estimate:", QURAN_WORD_ESTIMATE)

Sample ayahs: 37
Sample words: 296
Whole Quran word estimate: 77430


In [4]:
@dataclass(frozen=True)
class Chunk:
    chunk_size: int
    index: int
    target_text: str
    previous_text: str
    next_text: str
    target_refs: list[str]
    previous_refs: list[str]
    next_refs: list[str]
    target_word_count: int
    prompt_word_count: int


def unique_refs(words: list[dict[str, str]]) -> list[str]:
    refs: list[str] = []
    for item in words:
        if item["ref"] not in refs:
            refs.append(item["ref"])
    return refs


def words_to_text(words: list[dict[str, str]]) -> str:
    return " ".join(item["word"] for item in words)


def build_chunks(chunk_size: int, context_ratio: float = 0.35, max_chunks: int = 3) -> list[Chunk]:
    context_words = max(24, int(chunk_size * context_ratio))
    stride = chunk_size
    chunks: list[Chunk] = []
    for index, start in enumerate(range(0, len(flat_words), stride)):
        target = flat_words[start : start + chunk_size]
        if not target:
            break
        previous = flat_words[max(0, start - context_words) : start]
        next_ = flat_words[start + len(target) : start + len(target) + context_words]
        prompt_word_count = len(previous) + len(target) + len(next_)
        chunks.append(
            Chunk(
                chunk_size=chunk_size,
                index=index,
                target_text=words_to_text(target),
                previous_text=words_to_text(previous),
                next_text=words_to_text(next_),
                target_refs=unique_refs(target),
                previous_refs=unique_refs(previous),
                next_refs=unique_refs(next_),
                target_word_count=len(target),
                prompt_word_count=prompt_word_count,
            )
        )
        if len(chunks) >= max_chunks:
            break
    return chunks


benchmark_chunks = {size: build_chunks(size) for size in CHUNK_SIZES}
for size, chunks in benchmark_chunks.items():
    print(size, "words ->", len(chunks), "sample chunks; prompt words:", [c.prompt_word_count for c in chunks])

80 words -> 3 sample chunks; prompt words: [108, 136, 136]
160 words -> 2 sample chunks; prompt words: [216, 192]


## Prompt Contract

Model must emit one JSON object only. SurrealDB pipeline can map `source_refs`, `context_refs`, `categories`, `entities`, and `relations` into records plus `RELATE` edges.

In [5]:
SYSTEM_PROMPT = """You prepare Quran text chunks for insertion into SurrealDB knowledge graph.
Return exactly one JSON object. No markdown. No extra text.

Schema:
{
  "primary_category": "aqidah|ibadah|akhlaq|qisas|ahkam|duaa|language|other",
  "secondary_categories": ["string"],
  "entities": [{"name_ar": "string", "type": "person|place|group|concept|divine_name|scripture|other"}],
  "relations": [{"subject": "string", "predicate": "string", "object": "string"}],
  "summary_ar": "short Arabic summary",
  "surrealdb_notes": {
    "chunk_kind": "quran",
    "relation_strategy": "RELATE source_chunk->mentions/entity->entity and source_chunk->has_category->category"
  }
}

Rules:
- Analyze TARGET CHUNK only.
- Use PREVIOUS CONTEXT and NEXT CONTEXT only to preserve meaning.
- Keep relations concise and directly supported by target chunk.
- If unsure, return empty lists instead of invented facts.
"""


def build_prompt(chunk: Chunk) -> str:
    return f"""PREVIOUS CONTEXT refs={chunk.previous_refs}
{chunk.previous_text or "[none]"}

TARGET CHUNK refs={chunk.target_refs}
{chunk.target_text}

NEXT CONTEXT refs={chunk.next_refs}
{chunk.next_text or "[none]"}

Return JSON for TARGET CHUNK only."""

In [6]:
REQUIRED_TOP_LEVEL_KEYS = {
    "primary_category",
    "secondary_categories",
    "entities",
    "relations",
    "summary_ar",
    "surrealdb_notes",
}


def extract_json_object(text: str) -> dict[str, Any] | None:
    text = text.strip()
    if text.startswith("```"):
        text = re.sub(r"^```(?:json)?", "", text).strip()
        text = re.sub(r"```$", "", text).strip()
    try:
        return json.loads(text)
    except json.JSONDecodeError:
        match = re.search(r"\{.*\}", text, re.S)
        if not match:
            return None
        try:
            return json.loads(match.group(0))
        except json.JSONDecodeError:
            return None


def score_payload(payload: dict[str, Any] | None) -> dict[str, Any]:
    if payload is None:
        return {
            "json_valid": False,
            "required_key_count": 0,
            "entity_count": 0,
            "relation_count": 0,
            "quality_score": 0.0,
        }
    keys = set(payload)
    required_key_count = len(REQUIRED_TOP_LEVEL_KEYS & keys)
    entities = payload.get("entities") if isinstance(payload.get("entities"), list) else []
    relations = payload.get("relations") if isinstance(payload.get("relations"), list) else []
    primary = payload.get("primary_category")
    category_ok = primary in {"aqidah", "ibadah", "akhlaq", "qisas", "ahkam", "duaa", "language", "other"}
    quality_score = (
        0.45 * (required_key_count / len(REQUIRED_TOP_LEVEL_KEYS))
        + 0.20 * float(category_ok)
        + 0.20 * min(len(entities), 3) / 3
        + 0.15 * min(len(relations), 3) / 3
    )
    return {
        "json_valid": True,
        "required_key_count": required_key_count,
        "entity_count": len(entities),
        "relation_count": len(relations),
        "primary_category": primary,
        "quality_score": round(quality_score, 3),
    }

In [7]:
def generate_chunk_metadata(model: str, chunk: Chunk) -> dict[str, Any]:
    payload = {
        "model": model,
        "prompt": build_prompt(chunk),
        "system": SYSTEM_PROMPT,
        "stream": False,
        "format": "json",
        "keep_alive": "10m",
        "options": {
            "temperature": 0.1,
            "top_p": 0.9,
            "num_ctx": 4096,
        },
    }
    started = time.perf_counter()
    raw = ollama_post("/api/generate", payload)
    elapsed_seconds = time.perf_counter() - started
    response_text = raw.get("response", "")
    parsed = extract_json_object(response_text)
    score = score_payload(parsed)
    eval_count = raw.get("eval_count") or 0
    eval_duration = raw.get("eval_duration") or 0
    prompt_eval_count = raw.get("prompt_eval_count") or 0
    tokens_per_second = (eval_count / (eval_duration / 1_000_000_000)) if eval_count and eval_duration else None
    return {
        "model": model,
        "chunk_size": chunk.chunk_size,
        "chunk_index": chunk.index,
        "target_refs": chunk.target_refs,
        "previous_refs": chunk.previous_refs,
        "next_refs": chunk.next_refs,
        "target_word_count": chunk.target_word_count,
        "prompt_word_count": chunk.prompt_word_count,
        "elapsed_seconds": round(elapsed_seconds, 3),
        "eval_count": eval_count,
        "prompt_eval_count": prompt_eval_count,
        "tokens_per_second": round(tokens_per_second, 2) if tokens_per_second else None,
        "score": score,
        "parsed": parsed,
        "raw_response_preview": response_text[:500],
    }


def warm_model(model: str) -> None:
    ollama_post(
        "/api/generate",
        {
            "model": model,
            "prompt": "Return JSON: {\"ok\": true}",
            "stream": False,
            "format": "json",
            "keep_alive": "10m",
            "options": {"temperature": 0, "num_predict": 16},
        },
        timeout=MODEL_TIMEOUT_SECONDS,
    )

## Run Benchmark

This cell performs real Ollama calls. Default sweep is 4 models x 3 chunk sizes x up to 3 chunks. Reduce with env vars if needed:

`BENCHMARK_MODELS=llama3.2:3b,qwen2.5:7b`

`CHUNK_SIZES=80,160`

In [8]:
results: list[dict[str, Any]] = []

for model in models:
    print(f"\nWarm model: {model}")
    warm_model(model)
    for size in CHUNK_SIZES:
        for chunk in benchmark_chunks[size]:
            print(f"Run model={model} size={size} chunk={chunk.index} refs={chunk.target_refs}")
            try:
                result = generate_chunk_metadata(model, chunk)
                results.append(result)
                print(
                    "  seconds=", result["elapsed_seconds"],
                    "quality=", result["score"]["quality_score"],
                    "json=", result["score"]["json_valid"],
                    "cat=", result["score"].get("primary_category"),
                )
            except Exception as exc:
                error_result = {
                    "model": model,
                    "chunk_size": size,
                    "chunk_index": chunk.index,
                    "target_refs": chunk.target_refs,
                    "target_word_count": chunk.target_word_count,
                    "prompt_word_count": chunk.prompt_word_count,
                    "error": repr(exc),
                    "elapsed_seconds": None,
                    "score": {"json_valid": False, "quality_score": 0.0},
                }
                results.append(error_result)
                print("  ERROR", repr(exc))

results_path = OUTPUT_DIR / "ollama_quran_chunk_benchmark_results.json"
results_path.write_text(json.dumps(results, ensure_ascii=False, indent=2), encoding="utf-8")
print("Saved:", results_path)
print("Rows:", len(results))


Warm model: llama3.2:3b


Run model=llama3.2:3b size=80 chunk=0 refs=['1:1', '1:2', '1:3', '1:4', '1:5', '1:6', '1:7', '2:255', '2:285']


  seconds= 3.028 quality= 0.717 json= True cat= duaa
Run model=llama3.2:3b size=80 chunk=1 refs=['2:285', '2:286', '36:1', '36:2', '36:3']


  seconds= 5.72 quality= 0.817 json= True cat= duaa
Run model=llama3.2:3b size=80 chunk=2 refs=['36:3', '36:4', '36:5', '36:6', '36:7', '36:8', '36:9', '36:10', '36:11', '36:12', '112:1']


  seconds= 3.452 quality= 0.767 json= True cat= duaa
Run model=llama3.2:3b size=160 chunk=0 refs=['1:1', '1:2', '1:3', '1:4', '1:5', '1:6', '1:7', '2:255', '2:285', '2:286', '36:1', '36:2', '36:3']


...

  seconds= 14.072 quality= 1.0 json= True cat= aqidah
Run model=qwen2.5:7b size=80 chunk=1 refs=['2:285', '2:286', '36:1', '36:2', '36:3']


  seconds= 14.446 quality= 1.0 json= True cat= ibadah
Run model=qwen2.5:7b size=80 chunk=2 refs=['36:3', '36:4', '36:5', '36:6', '36:7', '36:8', '36:9', '36:10', '36:11', '36:12', '112:1']


  seconds= 10.375 quality= 0.95 json= True cat= ibadah
Run model=qwen2.5:7b size=160 chunk=0 refs=['1:1', '1:2', '1:3', '1:4', '1:5', '1:6', '1:7', '2:255', '2:285', '2:286', '36:1', '36:2', '36:3']


  seconds= 12.338 quality= 1.0 json= True cat= aqidah
Run model=qwen2.5:7b size=160 chunk=1 refs=['36:3', '36:4', '36:5', '36:6', '36:7', '36:8', '36:9', '36:10', '36:11', '36:12', '112:1', '112:2', '112:3', '112:4', '113:1', '113:2', '113:3', '113:4', '113:5', '114:1', '114:2', '114:3', '114:4', '114:5', '114:6']


  seconds= 18.518 quality= 1.0 json= True cat= aqidah
Saved: /app/notebooks/experiments/experiments/output/ollama_quran_chunk_benchmark_results.json
Rows: 10


In [9]:
def mean(values: list[float]) -> float | None:
    clean = [value for value in values if value is not None]
    return statistics.mean(clean) if clean else None


summary_rows: list[dict[str, Any]] = []
for model in models:
    for size in CHUNK_SIZES:
        subset = [
            row for row in results
            if row.get("model") == model and row.get("chunk_size") == size and not row.get("error")
        ]
        if not subset:
            summary_rows.append({"model": model, "chunk_size": size, "runs": 0, "error": "no successful runs"})
            continue
        avg_seconds = mean([row["elapsed_seconds"] for row in subset])
        avg_quality = mean([row["score"]["quality_score"] for row in subset])
        json_rate = sum(1 for row in subset if row["score"]["json_valid"]) / len(subset)
        avg_entities = mean([row["score"]["entity_count"] for row in subset])
        avg_relations = mean([row["score"]["relation_count"] for row in subset])
        quran_chunks = math.ceil(QURAN_WORD_ESTIMATE / size)
        estimated_total_seconds = quran_chunks * avg_seconds
        summary_rows.append(
            {
                "model": model,
                "chunk_size": size,
                "runs": len(subset),
                "avg_seconds_per_chunk": round(avg_seconds, 3),
                "avg_quality_score": round(avg_quality, 3),
                "json_valid_rate": round(json_rate, 3),
                "avg_entities": round(avg_entities, 2),
                "avg_relations": round(avg_relations, 2),
                "estimated_quran_chunks": quran_chunks,
                "estimated_total_minutes": round(estimated_total_seconds / 60, 1),
                "estimated_total_hours": round(estimated_total_seconds / 3600, 2),
            }
        )

summary_rows = sorted(
    summary_rows,
    key=lambda row: (
        -row.get("avg_quality_score", 0),
        row.get("estimated_total_minutes", 10**9),
        -row.get("chunk_size", 0),
    ),
)

summary_path = OUTPUT_DIR / "ollama_quran_chunk_benchmark_summary.json"
summary_path.write_text(json.dumps(summary_rows, ensure_ascii=False, indent=2), encoding="utf-8")
print(json.dumps(summary_rows, ensure_ascii=False, indent=2))
print("Saved:", summary_path)

[
  {
    "model": "qwen2.5:7b",
    "chunk_size": 160,
    "runs": 2,
    "avg_seconds_per_chunk": 15.428,
    "avg_quality_score": 1.0,
    "json_valid_rate": 1.0,
    "avg_entities": 5,
    "avg_relations": 4.5,
    "estimated_quran_chunks": 484,
    "estimated_total_minutes": 124.5,
    "estimated_total_hours": 2.07
  },
  {
    "model": "qwen2.5:7b",
    "chunk_size": 80,
    "runs": 3,
    "avg_seconds_per_chunk": 12.964,
    "avg_quality_score": 0.983,
    "json_valid_rate": 1.0,
    "avg_entities": 4.67,
    "avg_relations": 3.67,
    "estimated_quran_chunks": 968,
    "estimated_total_minutes": 209.2,
    "estimated_total_hours": 3.49
  },
  {
    "model": "llama3.2:3b",
    "chunk_size": 160,
    "runs": 2,
    "avg_seconds_per_chunk": 4.675,
    "avg_quality_score": 0.792,
    "json_valid_rate": 1.0,
    "avg_entities": 1,
    "avg_relations": 2.5,
    "estimated_quran_chunks": 484,
    "estimated_total_minutes": 37.7,
    "estimated_total_hours": 0.63
  },
  {
    "model": 

In [10]:
best = summary_rows[0]
best_same_model = [
    row for row in summary_rows
    if row["model"] == best["model"] and row.get("runs", 0) > 0
]
best_size_for_model = sorted(
    best_same_model,
    key=lambda row: (-row["avg_quality_score"], row["estimated_total_minutes"])
)[0]

print("Best overall:", best)
print("Recommended first pass:", best_size_for_model)

Best overall: {'model': 'qwen2.5:7b', 'chunk_size': 160, 'runs': 2, 'avg_seconds_per_chunk': 15.428, 'avg_quality_score': 1.0, 'json_valid_rate': 1.0, 'avg_entities': 5, 'avg_relations': 4.5, 'estimated_quran_chunks': 484, 'estimated_total_minutes': 124.5, 'estimated_total_hours': 2.07}
Recommended first pass: {'model': 'qwen2.5:7b', 'chunk_size': 160, 'runs': 2, 'avg_seconds_per_chunk': 15.428, 'avg_quality_score': 1.0, 'json_valid_rate': 1.0, 'avg_entities': 5, 'avg_relations': 4.5, 'estimated_quran_chunks': 484, 'estimated_total_minutes': 124.5, 'estimated_total_hours': 2.07}


## SurrealDB Mapping Draft

Recommended persisted shape:
- `source_chunk`: target Quran text, refs, chunk index, target word count, model metadata, category confidence;
- `context_window`: previous/next refs and text hashes, not duplicated full Quran if canonical ayah table exists;
- `category`: stable taxonomy nodes such as `aqidah`, `duaa`, `ibadah`;
- `entity`: divine names, scripture concepts, people/groups/places;
- edges: `RELATE source_chunk->has_category->category`, `RELATE source_chunk->mentions->entity`, `RELATE entity->related_to->entity` for model-supported relations.

Keep prompt context outside target metadata so neighboring ayahs improve meaning without polluting target refs.

In [11]:
def markdown_table(rows: list[dict[str, Any]]) -> str:
    headers = [
        "model",
        "chunk_size",
        "runs",
        "avg_seconds_per_chunk",
        "avg_quality_score",
        "json_valid_rate",
        "avg_entities",
        "avg_relations",
        "estimated_quran_chunks",
        "estimated_total_minutes",
        "estimated_total_hours",
    ]
    lines = [
        "| " + " | ".join(headers) + " |",
        "| " + " | ".join(["---"] * len(headers)) + " |",
    ]
    for row in rows:
        lines.append("| " + " | ".join(str(row.get(header, "")) for header in headers) + " |")
    return "\n".join(lines)


report = f"""# Ollama Quran Chunk Benchmark Findings

Generated by `OpenBayanBackend/notebooks/experiments/ollama_quran_chunk_benchmark.ipynb`.

## Setup

- Runtime: Docker Compose `jupyter` service.
- Ollama URL: configured through `OLLAMA_URL` / `OLLAMA_HOST`.
- Whole Quran estimate: {QURAN_WORD_ESTIMATE:,} Arabic words.
- Models tested: {", ".join(models)}.
- Target chunk sizes: {", ".join(str(size) for size in CHUNK_SIZES)} words.
- Context: previous + next window included in prompt, output applies to target only.

## Results

{markdown_table(summary_rows)}

## Recommendation

Use `{best_size_for_model["model"]}` with `{best_size_for_model["chunk_size"]}` target words for first ingestion pass.

Reason: best observed quality/speed balance in this sample. Estimated full Quran runtime is about `{best_size_for_model["estimated_total_minutes"]}` minutes (`{best_size_for_model["estimated_total_hours"]}` hours) for one sequential worker.

## Pipeline Notes

- Store canonical Quran ayahs separately; store chunk refs and context refs on chunk records.
- Keep neighboring context in prompt only, unless UI/debugging needs context snapshots.
- Persist model output as structured JSON plus raw preview for audit.
- Use SurrealDB relations:
  - `RELATE source_chunk->has_category->category`
  - `RELATE source_chunk->mentions->entity`
  - `RELATE source_chunk->supports_relation->relation`
- Rerun benchmark against full representative samples before production ingest.
"""

report_path = OUTPUT_DIR / "ollama_quran_chunk_benchmark_findings.md"
report_path.write_text(report, encoding="utf-8")
print(report)
print("Saved:", report_path)

# Ollama Quran Chunk Benchmark Findings

Generated by `OpenBayanBackend/notebooks/experiments/ollama_quran_chunk_benchmark.ipynb`.

## Setup

- Runtime: Docker Compose `jupyter` service.
- Ollama URL: configured through `OLLAMA_URL` / `OLLAMA_HOST`.
- Whole Quran estimate: 77,430 Arabic words.
- Models tested: llama3.2:3b, qwen2.5:7b.
- Target chunk sizes: 80, 160 words.
- Context: previous + next window included in prompt, output applies to target only.

## Results

| model | chunk_size | runs | avg_seconds_per_chunk | avg_quality_score | json_valid_rate | avg_entities | avg_relations | estimated_quran_chunks | estimated_total_minutes | estimated_total_hours |
| --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- |
| qwen2.5:7b | 160 | 2 | 15.428 | 1.0 | 1.0 | 5 | 4.5 | 484 | 124.5 | 2.07 |
| qwen2.5:7b | 80 | 3 | 12.964 | 0.983 | 1.0 | 4.67 | 3.67 | 968 | 209.2 | 3.49 |
| llama3.2:3b | 160 | 2 | 4.675 | 0.792 | 1.0 | 1 | 2.5 | 484 | 37.7 | 0.63 |
| llama3.2:3b | 80 | 3 | 

## Next Steps

- Increase sample coverage: Makki/Madani, legal verses, narratives, short surahs, long ayahs.
- Add human-scored gold labels for category/relation precision.
- Test parallel workers and Ollama `keep_alive` behavior before production run.
- Promote stable chunking + Ollama helper code into reusable worker modules.